# Draft A: results, tables and figures

Reads the finished runs under `runs/A`, and
writes `results_tables.tex` and figures into `papers/admissible_pooling/`.
Run `notebooks/train.ipynb` and `scripts/swap_eval.py` first.

In [1]:
import os, sys, glob, json
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path[:0] = [ROOT, os.path.join(ROOT, 'scripts')]
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# IGNN_RUNS / IGNN_OUT only redirect outputs for testing; leave them unset for real runs.
RUNS = os.environ.get('IGNN_RUNS', os.path.join(ROOT, 'runs'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('root:', ROOT, '| runs:', RUNS, '| device:', DEVICE)

OUT = os.environ.get('IGNN_OUT', os.path.join(ROOT, 'papers', 'A_admissible_pooling'))
os.makedirs(os.path.join(OUT, 'figures'), exist_ok=True)
from collect import collect, summarize

root: e:\university\Projects\Bachlour | runs: e:\university\Projects\Bachlour\runs | device: cuda


In [2]:
tests = collect('A', RUNS)
if tests.empty:
    print('no finished runs under', os.path.join(RUNS, 'A'))
else:
    tests['op'] = tests['name'].str.split('_').str[0]
    tests['layout'] = tests['name'].str.split('_').str[1]
    summary = summarize(tests, keys=('dataset', 'name', 'eps'))
    display(summary.round(4))
    # the unmodified CNN-7 against the published IBP numbers (CTBench, Table 1: clean, certified)
    PUBLISHED = {('mnist', 0.1): (98.87, 98.26), ('mnist', 0.3): (98.54, 93.80),
                 ('cifar10', 2 / 255): (67.49, 55.99), ('cifar10', 8 / 255): (48.51, 35.28)}
    for (ds, e), (cl, ce) in PUBLISHED.items():
        r = summary[(summary.dataset == ds) & summary.name.str.startswith('ref_') & np.isclose(summary.eps, e)]
        if len(r):
            print(f"{ds} eps={e:.4f}: ours clean {r.clean_acc_mean.iloc[0] * 100:.2f} / certified "
                  f"{r.verified_acc_mean.iloc[0] * 100:.2f}, published {cl} / {ce}  ({int(r.n_seeds.iloc[0])} seeds)")

,dataset,name,eps,n_seeds,clean_acc_mean,clean_acc_std,clean_acc_ci95,verified_acc_mean,verified_acc_std,verified_acc_ci95
0,cifar10,lattice_pool1_eps2,0.0000,3,0.6744,0.0021,0.0052,0.6744,0.0021,0.0052
1,cifar10,lattice_pool1_eps2,0.0078,3,0.6744,0.0021,0.0052,0.5308,0.0064,0.0159
2,cifar10,lattice_pool1_eps8,0.0000,3,0.4813,0.0052,0.0129,0.4813,0.0052,0.0129
3,cifar10,lattice_pool1_eps8,0.0314,3,0.4813,0.0052,0.0129,0.3411,0.0033,0.0082
4,cifar10,lattice_pool2_eps2,0.0000,3,0.6669,0.0023,0.0058,0.6669,0.0023,0.0058
...,...,...,...,...,...,...,...,...,...,...
59,mnist,ref_cnn7_eps03,0.3000,3,0.9849,0.0005,0.0013,0.9322,0.0010,0.0025
60,mnist,std_pool1_eps01,0.0000,3,0.9956,0.0004,0.0010,0.9956,0.0004,0.0010
61,mnist,std_pool1_eps01,0.1000,3,0.9956,0.0004,0.0010,0.0000,0.0000,0.0000
62,mnist,std_pool1_eps03,0.0000,3,0.9958,0.0002,0.0006,0.9958,0.0002,0.0006


mnist eps=0.1000: ours clean 99.16 / certified 96.51, published 98.87 / 98.26  (3 seeds)
mnist eps=0.3000: ours clean 98.49 / certified 93.22, published 98.54 / 93.8  (3 seeds)
cifar10 eps=0.0078: ours clean 67.05 / certified 52.83, published 67.49 / 55.99  (3 seeds)
cifar10 eps=0.0314: ours clean 47.87 / certified 33.92, published 48.51 / 35.28  (3 seeds)


In [3]:
swaps = []
for f in glob.glob(os.path.join(RUNS, 'A', '*', '*', 'seed*', 'swap_results.csv')):
    d = pd.read_csv(f)
    parts = os.path.normpath(f).split(os.sep)
    d['dataset'], d['name'], d['seed'] = parts[-4], parts[-3], int(parts[-2][4:])
    d['layout'] = d['name'].str.split('_').str[1]
    d['tag'] = d['name'].str.split('_').str[2].fillna('old')     # 'old': FMNIST, earlier recipe
    swaps.append(d)
swaps = pd.concat(swaps, ignore_index=True) if swaps else pd.DataFrame()
if not swaps.empty:
    agg = swaps.groupby(['dataset', 'tag', 'layout', 'trained_with', 'op']).agg(
        n_seeds=('seed', 'nunique'), violation=('violation_rate', 'mean'), violation_max=('violation_rate', 'max'),
        clean=('clean_acc', 'mean'), verified=('verified_acc', 'mean'), verified_std=('verified_acc', 'std'),
        width_ratio=('width_ratio', 'mean')).reset_index()
    display(agg.round(4))

,dataset,tag,layout,trained_with,op,n_seeds,violation,violation_max,clean,verified,verified_std,width_ratio
0,cifar10,eps2,pool1,lattice,lattice,3,0.0,0.0,0.6744,0.5308,0.0064,1.0000
1,cifar10,eps2,pool1,lattice,lex1,3,0.0,0.0,0.6744,0.5315,0.0063,0.9943
2,cifar10,eps2,pool1,lattice,lex2,3,0.0,0.0,0.6744,0.5297,0.0060,1.0085
3,cifar10,eps2,pool1,lattice,xuyager,3,0.0,0.0,0.6744,0.5309,0.0066,0.9997
4,cifar10,eps2,pool1,lex2,lattice,3,0.0,0.0,0.6734,0.5275,0.0005,1.0000
...,...,...,...,...,...,...,...,...,...,...,...,...
91,mnist,eps03,pool2,lattice,xuyager,3,0.0,0.0,0.9772,0.9164,0.0021,0.9989
92,mnist,eps03,pool2,lex2,lattice,3,0.0,0.0,0.9698,0.9045,0.0034,1.0000
93,mnist,eps03,pool2,lex2,lex1,3,0.0,0.0,0.9698,0.9051,0.0028,0.9927
94,mnist,eps03,pool2,lex2,lex2,3,0.0,0.0,0.9698,0.8934,0.0048,1.0958


## Table `tab:main`

Every trained network evaluated with all four operators on the same weights (H1-H4), on the
lattice-trained and the Lex2-trained weights. Bold marks the best clean and verified accuracy per
benchmark; verified accuracy is reported only for enclosing operators.

In [ ]:
def pm(s, scale=100, bold=False):
    s = s.dropna()
    if len(s) == 0:
        return '--'
    body = f'{s.mean() * scale:.2f}\\pm{(s.std() if len(s) > 1 else 0) * scale:.2f}'
    return f'$\\mathbf{{{body}}}$' if bold else f'${body}$'


def rate(v):
    """0 -> $0$, otherwise mantissa times a power of ten, in LaTeX."""
    if v == 0:
        return '$0$'
    m, e = f'{v:.1e}'.split('e')
    return f'${m}\\cdot10^{{{int(e)}}}$'


BENCHMARKS = [('mnist', 'eps01', 'MNIST $0.1$'), ('mnist', 'eps03', 'MNIST $0.3$'),
              ('cifar10', 'eps2', 'CIFAR-10 $2/255$'), ('cifar10', 'eps8', 'CIFAR-10 $8/255$')]
FMNIST = [('fmnist', 'old', 'Fashion-MNIST $0.1$')]          # earlier recipe: its own table, in the appendix
# (weights trained with, operator evaluated): every operator on the lattice weights, then the Lex2 weights
ROWS = [('lattice', 'lattice'), ('lattice', 'lex2'), ('lattice', 'lex1'), ('lattice', 'xuyager'),
        ('lex2', 'lex2'), ('lex2', 'lattice')]
NAMES = {'lattice': 'Lattice', 'lex2': 'Lex2', 'lex1': 'Lex1', 'xuyager': 'Xu--Yager'}
SOUND = ('lattice', 'lex2')
# the journal asks for a short caption and for the notes below the table body
NOTES = ('Window viol.\\ and logit viol.: fractions of the windows of the first pooling layer and of the (input, logit) pairs '
         'whose sampled values leave the computed interval by more than $10^{-4}$ and $10^{-5}$. Max excess: largest distance of a '
         'sampled window maximum outside its interval, over seeds. Verified: verified accuracy in \\%, mean $\\pm$ standard '
         'deviation over three seeds, reported only for enclosing operators; bold marks the best per benchmark. Width ratio: mean '
         'width of the predicted logit relative to the lattice operator on the same weights. Clean accuracy does not depend on '
         'the operator and is given in ')


def main_table(swaps, layout, benchmarks, label, caption, clean_in):
    """One table of the same-weights evaluation; bold marks the best verified accuracy per benchmark (enclosing operators only)."""
    lines = ['\\begin{table}[tbp]', '\\centering\\footnotesize', '\\setlength{\\tabcolsep}{3.5pt}',
             f'\\caption{{{caption}}}', f'\\label{{{label}}}', '\\begin{tabular}{llccccc}', '\\toprule',
             'Weights & Pooling & Window viol. & Max excess & Logit viol. & Verified & Width ratio \\\\',
             '\\midrule']
    for ds, tag, ds_label in benchmarks:
        d = swaps[(swaps.dataset == ds) & (swaps.tag == tag) & (swaps.layout == layout)]
        if d.empty:
            continue
        groups = {(tw, op): d[(d.trained_with == tw) & (d.op == op)] for tw, op in ROWS}
        groups = {k: g for k, g in groups.items() if not g.empty}
        best_ver = max(round(g.verified_acc.mean() * 100, 2) for (tw, op), g in groups.items() if op in SOUND)
        # the benchmark is a heading row, so that the table fits the width of the text
        lines += [f'\\multicolumn{{7}}{{l}}{{\\emph{{{ds_label}}}}} \\\\', '\\addlinespace[2pt]']
        for (tw, op), g in groups.items():
            wv = rate(g.window_violation_rate.mean()) if 'window_violation_rate' in g else '--'
            mx = rate(g.max_window_excess.max()) if 'max_window_excess' in g else '--'
            ver = pm(g.verified_acc, bold=round(g.verified_acc.mean() * 100, 2) == best_ver) if op in SOUND else 'n/a'
            lines.append(f"{NAMES[tw]} & {NAMES[op]} & {wv} & {mx} & {rate(g.violation_rate.mean())} "
                         f"& {ver} & ${g.width_ratio.mean():.3f}$ \\\\")
        lines.append('\\midrule')
    lines[-1] = '\\bottomrule'
    lines += ['\\end{tabular}', '\\par\\smallskip', f'\\parbox{{\\linewidth}}{{\\footnotesize {NOTES}\\Cref{{{clean_in}}}.}}',
              '\\end{table}']
    return '\n'.join(lines)


if not swaps.empty:
    head = '% generated by notebooks/A_analysis.ipynb; do not edit by hand\n'
    same = 'evaluated with every pooling operator on the same weights.'
    tex = head + main_table(swaps, 'pool1', BENCHMARKS, 'tab:main', f'Each trained network with one pooling layer {same}', 'tab:train') + '\n\n' \
        + main_table(swaps, 'pool2', BENCHMARKS, 'tab:main-pool2', f'Each trained network with two pooling layers {same}', 'tab:train') + '\n'
    with open(os.path.join(OUT, 'results_tables.tex'), 'w', encoding='utf-8') as f:
        f.write(tex)
    print(tex)
    tex = head + main_table(swaps, 'pool1', FMNIST, 'tab:main-fmnist',
                            f'Fashion-MNIST networks of the earlier recipe (one pooling layer) {same}', 'tab:fmnist') + '\n'
    with open(os.path.join(OUT, 'results_table_fmnist.tex'), 'w', encoding='utf-8') as f:
        f.write(tex)
    print(tex)

## Figures

The figures of the paper (`Figure_3.pdf`, `Figure_4.pdf`, `Figure_5.pdf`) are drawn at their printed size by
`figures/make_figures.py` in the paper folder; `Figure_1` and `Figure_2` are TikZ drawings next to it.
The cell below gives quick previews of the same data:

* `width_ratio.pdf`: width ratio Lex2/lattice per dataset and layout (H3: at least 1).
* `stability.pdf`: validation verified accuracy and gradient norm, lattice vs Lex2 training (H5).
* `violation_sim.pdf` is written by `scripts/A_violation_sim.py`; its summary is shown below.

In [ ]:
LABELS = {'cifar10 eps2': 'CIFAR-10 2/255', 'cifar10 eps8': 'CIFAR-10 8/255', 'fmnist old': 'FMNIST 0.1*',
          'mnist eps01': 'MNIST 0.1', 'mnist eps03': 'MNIST 0.3'}
# one tick per benchmark (in the order of tab:main), the layouts side by side
LAYOUTS = {'pool1': ('1 pooling layer', 'C0', -0.2), 'pool2': ('2 pooling layers', 'C1', 0.2)}

if not swaps.empty:
    d = swaps[(swaps.trained_with == 'lattice') & (swaps.op == 'lex2')]
    benches = [(ds, tag) for ds, tag, _ in BENCHMARKS + FMNIST if ((d.dataset == ds) & (d.tag == tag)).any()]
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    for layout, (lab, col, dx) in LAYOUTS.items():
        pos = [i + dx for i, (ds, tag) in enumerate(benches) if ((d.dataset == ds) & (d.tag == tag) & (d.layout == layout)).any()]
        data = [d[(d.dataset == ds) & (d.tag == tag) & (d.layout == layout)].width_ratio for ds, tag in benches
                if ((d.dataset == ds) & (d.tag == tag) & (d.layout == layout)).any()]
        if data:
            bp = ax.boxplot(data, positions=pos, widths=0.32, patch_artist=True, manage_ticks=False,
                            boxprops=dict(facecolor=col, alpha=0.35, edgecolor=col), medianprops=dict(color=col),
                            whiskerprops=dict(color=col), capprops=dict(color=col), flierprops=dict(markeredgecolor=col))
            bp['boxes'][0].set_label(lab)
    ax.set_xticks(range(len(benches)), [LABELS.get(f'{ds} {tag}', f'{ds} {tag}') for ds, tag in benches])
    ax.axhline(1, color='k', ls=':', lw=1)
    ax.set(ylabel='width ratio Lex2 / lattice', title='same (lattice-trained) weights')
    ax.legend(fontsize=8, loc='upper left')
    fig.tight_layout(); fig.savefig(os.path.join(OUT, 'figures', 'width_ratio.pdf')); plt.show()

curves = []
for f in glob.glob(os.path.join(RUNS, 'A', '*', '*', 'seed*', 'metrics.csv')):
    parts = os.path.normpath(f).split(os.sep)
    m = pd.read_csv(f).assign(dataset=parts[-4], name=parts[-3], seed=parts[-2])
    m['bench'] = m.dataset + ' ' + (parts[-3].split('_') + ['old'])[2]
    m['op_layout'] = '_'.join(parts[-3].split('_')[:2])
    curves.append(m)
curves = pd.concat(curves) if curves else pd.DataFrame(columns=['bench', 'op_layout'])
# only datasets with IBP runs so far; a dataset that has just std runs would leave an empty log axis
dss = sorted(curves[curves.op_layout.isin(['lattice_pool1', 'lex2_pool1'])].bench.unique())
if dss:
    fig, axes = plt.subplots(2, len(dss), figsize=(4.5 * len(dss), 6), squeeze=False)
    for j, ds in enumerate(dss):
        for name, col in [('lattice_pool1', 'C1'), ('lex2_pool1', 'C2')]:
            g = curves[(curves.bench == ds) & (curves.op_layout == name)].groupby('epoch')
            for i, v in enumerate(['val_verified', 'grad_norm']):
                mu, sd = g[v].mean(), g[v].std().fillna(0)
                axes[i, j].plot(mu.index, mu, color=col, label=name)
                axes[i, j].fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=.2)
        axes[0, j].set(title=LABELS.get(ds, ds), ylabel='monitored verified acc'); axes[1, j].set(ylabel='grad norm', xlabel='epoch', yscale='log')
        axes[0, j].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, 'figures', 'stability.pdf')); plt.show()

sim = os.path.join(ROOT, 'papers', 'A_admissible_pooling', 'results', 'violation_sim_summary.csv')
if os.path.exists(sim):
    display(pd.read_csv(sim).round(5))
else:
    print('run: python scripts/A_violation_sim.py')